# DLinear 후속 실험: Table 8 시간·메모리 + 미해결 칸
`DLinear_extra_experiments.ipynb` 다음에 돌리는 노트북.

| 순서 | 실험 | 확인하는 것 | 예상 시간 (T4) |
|---|---|---|---|
| 1 | Table 8 효율 | Electricity, L=96, T=720에서 추론 시간과 GPU 메모리 | 1분 |
| 2 | 미해결 칸 재실행 | Exchange 720 · ILI 24~60 · ETTh2 720 (다변량)을 2022년 5월 스크립트·코드로 다시 돌리기 | 20~30분 |

**실행:** 서버는 **GPU → T4** (High-RAM 필요 없음) → **Run All**.

**실험 2의 배경.** 공식 레포 이력을 보면 2022-08-15 커밋 `7f7a25b`에서 두 가지가 바뀌었다. 논문 결과표는 그 전에 만들어졌을 가능성이 있다.
- 스크립트 학습률: Exchange 720은 0.005 → 0.0005, ILI는 0.05 → 0.01 (ETTh2는 0.05 그대로)
- 모델 초기화: 선형층 가중치를 $1/L$로 채우던 두 줄이 주석 처리됨 (지금은 PyTorch 기본 무작위 초기화)

그래서 세 가지 설정을 `--itr 5`로 5번씩 돌려 비교한다. test 지표는 지금 코드(`drop_last=False`)와 논문 당시 코드(`drop_last=True`) 두 방식으로 함께 계산한다.

| 설정 | 학습률 | 초기화 |
|---|---|---|
| A. 지금 | 지금 스크립트 | 기본 |
| B. 2022-05 학습률 | Exchange 720: 0.005, ILI: 0.05 | 기본 |
| C. 2022-05 학습률 + 코드 | B와 같음 (ETTh2는 지금과 같음) | $1/L$ |

In [ ]:
# 0. GPU 확인
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
# 1. 코드·패치·데이터 (이미 있으면 건너뜀)
import os, subprocess
os.chdir('/content')
if not os.path.exists('LTSF-Linear'):
    subprocess.run(['git', 'clone', '-q', 'https://github.com/cure-lab/LTSF-Linear.git'], check=True)
os.chdir('/content/LTSF-Linear')

def patch(path, old, new):
    s = open(path).read()
    if new in s:
        return
    assert old in s, (path, old)
    open(path, 'w').write(s.replace(old, new))

patch('utils/tools.py', 'np.Inf', 'np.inf')                       # NumPy 2 호환 (필수)
patch('run_longExp.py', "default=10, help='data loader num workers'",
      "default=2, help='data loader num workers'")                  # Colab CPU 수에 맞춤 (결과 영향 없음)
patch('exp/exp_main.py', '                inputx.append(batch_x.detach().cpu().numpy())',
      '                pass  # inputx 미사용 (메모리 절약 패치)')
patch('exp/exp_main.py', '        inputx = np.concatenate(inputx, axis=0)', '        inputx = None')
# 같은 학습 결과로 논문 당시 평가 방식(test 마지막 덜 찬 배치 버림)도 같이 계산해서 출력. 원래 mse 줄은 그대로.
patch('exp/exp_main.py', "        print('mse:{}, mae:{}'.format(mse, mae))\n",
      "        print('mse:{}, mae:{}'.format(mse, mae))\n"
      "        _k = (len(preds) // self.args.batch_size) * self.args.batch_size\n"
      "        _m = metric(preds[:_k], trues[:_k])\n"
      "        print('droplast N:{} keep:{} mse_dl:{}, mae_dl:{}'.format(len(preds), _k, _m[1], _m[0]))\n")

os.makedirs('dataset', exist_ok=True)
if not os.path.exists('dataset/ETTh2.csv'):
    subprocess.run(['wget', '-q', '-O', 'dataset/ETTh2.csv',
                    'https://raw.githubusercontent.com/zhouhaoyi/ETDataset/main/ETT-small/ETTh2.csv'], check=True)
need = ['electricity', 'exchange_rate', 'national_illness']
if not all(os.path.exists(f'dataset/{f}.csv') for f in need):
    subprocess.run('pip -q install gdown && gdown -q --folder '
                   'https://drive.google.com/drive/folders/1ZOYpTUa82_jCcxIdTmyr0LXQfvaM9vIy -O raw_data && '
                   'find raw_data -name "*.csv" -exec cp -n {} dataset/ \\;', shell=True, check=True)
!ls -lh dataset

In [ ]:
%%writefile /content/lab.py
"""DLinear 추가 실험 도우미.

공식 스크립트(scripts/EXP-*/...sh)를 '가짜 python'으로 한 번 돌려서 실제 실행될 명령 목록만 뽑고,
그 명령을 하나씩 실행한다. 이렇게 하면
  - 하이퍼파라미터를 손으로 옮겨 적지 않고 공식 스크립트 값을 그대로 쓰고
  - 실험마다 진행 상황이 한 줄씩 찍히고
  - 중간에 끊겨도 다시 실행하면 끝난 실험은 건너뛴다.
"""
import json, os, re, shutil, subprocess, sys, time

import pandas as pd

ROOT = '/content/LTSF-Linear'
_SHIM = '/content/_shim'
_WORK = '/content/_collect'

DATASETS = {'etth1.csv': 'ETTh1', 'etth2.csv': 'ETTh2', 'ettm1.csv': 'ETTm1', 'ettm2.csv': 'ETTm2',
            'electricity.csv': 'Electricity', 'exchange_rate.csv': 'Exchange', 'traffic.csv': 'Traffic',
            'weather.csv': 'Weather', 'national_illness.csv': 'ILI'}


def commands(script, edits=()):
    """공식 스크립트에서 실행될 python 명령 목록을 뽑는다. edits = [(찾을 문자열, 바꿀 문자열), ...]"""
    os.makedirs(_SHIM, exist_ok=True)
    with open(f'{_SHIM}/python', 'w') as f:
        f.write('#!/usr/bin/env python3\n'
                'import json, os, sys\n'
                'try:\n    log = os.readlink("/proc/self/fd/1")\nexcept OSError:\n    log = ""\n'
                'with open(os.environ["CMD_FILE"], "a") as f:\n'
                '    f.write(json.dumps({"argv": sys.argv[1:], "log": log}) + "\\n")\n')
    os.chmod(f'{_SHIM}/python', 0o755)

    shutil.rmtree(_WORK, ignore_errors=True)
    for d in ['logs/LongForecasting/univariate', 'logs/LookBackWindow']:  # 스크립트가 안 만드는 폴더도 미리
        os.makedirs(f'{_WORK}/{d}')
    src = open(os.path.join(ROOT, script)).read()
    for a, b in edits:
        assert a in src, f'{a!r} 가 {script} 에 없음'
        src = src.replace(a, b)
    open(f'{_WORK}/run.sh', 'w').write(src)
    env = dict(os.environ, PATH=f'{_SHIM}:{os.environ["PATH"]}', CMD_FILE=f'{_WORK}/cmds.jsonl')
    subprocess.run(['bash', 'run.sh'], cwd=_WORK, env=env, check=True, stdout=subprocess.DEVNULL)
    out = [json.loads(l) for l in open(f'{_WORK}/cmds.jsonl')]
    for c in out:
        c['log'] = os.path.basename(c['log']) if c['log'].startswith(_WORK) else ''
    return out


def arg(c, *names, default=None):
    a = c['argv']
    for i, x in enumerate(a[:-1]):
        if x in names:
            return a[i + 1]
    return default


def set_arg(c, name, value):
    a = c['argv']
    if name in a:
        a[a.index(name) + 1] = str(value)
    else:
        a += [name, str(value)]
    return c


def info(c):
    return dict(model=arg(c, '--model'), dataset=DATASETS[arg(c, '--data_path').lower()],
                features=arg(c, '--features', '--feature', default='M'),
                seq_len=int(arg(c, '--seq_len')), pred_len=int(arg(c, '--pred_len')),
                individual='--individual' in c['argv'])


def parse_log(path):
    """로그에서 (testing 설정, mse, mae)를 순서대로 뽑는다. --itr N이면 N개."""
    if not os.path.exists(path):
        return []
    text = open(path, errors='ignore').read()
    out = []
    for m in re.finditer(r'>>>>>>>testing : (\S+?)<<<.*?mse:([-\d.eE+]+), mae:([-\d.eE+]+)', text, re.S):
        out.append((m.group(1), float(m.group(2)), float(m.group(3))))
    return out


def run(cmds, tag):
    """명령을 차례로 실행. 로그는 logs/<tag>/ 에 저장, 이미 끝난 실험은 건너뜀."""
    logdir = os.path.join(ROOT, 'logs', tag)
    os.makedirs(logdir, exist_ok=True)
    t0 = time.time()
    for i, c in enumerate(cmds, 1):
        c['logpath'] = os.path.join(logdir, c.get('name') or c['log'] or f'run{i}.log')
        d = info(c)
        label = (f"{d['dataset']} {d['features']} L={d['seq_len']} T={d['pred_len']}"
                 + (' individual' if d['individual'] else ''))
        need = int(arg(c, '--itr', default=1))
        if len(parse_log(c['logpath'])) >= need:
            print(f'[{i}/{len(cmds)}] {label}: 이미 완료, 건너뜀', flush=True)
            continue
        t = time.time()
        with open(c['logpath'], 'w') as f:
            r = subprocess.run([sys.executable] + c['argv'], cwd=ROOT, stdout=f, stderr=subprocess.STDOUT)
        subprocess.run('rm -f results/*/pred.npy', shell=True, cwd=ROOT)  # 예측 배열은 안 쓰므로 디스크 절약
        res = parse_log(c['logpath'])
        if r.returncode != 0 or len(res) < need:
            tail = open(c['logpath'], errors='ignore').read()[-600:]
            print(f'[{i}/{len(cmds)}] {label}: 실패 (exit {r.returncode})\n{tail}', flush=True)
            continue
        mses = ', '.join(f'{m:.4f}' for _, m, _ in res)
        print(f'[{i}/{len(cmds)}] {label}: mse {mses}  ({time.time() - t:.0f}s, 누적 {(time.time() - t0) / 60:.0f}분)',
              flush=True)


def collect(cmds):
    rows = []
    for c in cmds:
        for k, (setting, mse, mae) in enumerate(parse_log(c.get('logpath', ''))):
            rows.append(dict(info(c), run=k, mse=mse, mae=mae))
    return pd.DataFrame(rows)

## 실험 1. 논문 Table 8 DLinear 행 (Electricity, L=96, T=720)
논문 값: MACs 0.04G, 파라미터 139.7K, 시간 0.4ms, 메모리 687MiB (Titan XP, 시간은 5회 평균).
논문은 배치 크기와 메모리 측정 방식을 적지 않아서 batch 1·16·32를 모두 잰다. 메모리는 텐서가 실제로 쓴 양(peak)과 CUDA context를 포함한 GPU 전체 사용량(nvidia-smi) 두 가지를 본다.

In [ ]:
# 논문 Table 8 DLinear 행: Electricity, L=96, T=720 (변수 321개)
import os, subprocess, sys, time, types, statistics
if not os.path.isdir('/content/LTSF-Linear'):
    subprocess.run(['git', 'clone', '-q', 'https://github.com/cure-lab/LTSF-Linear.git', '/content/LTSF-Linear'], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'ptflops'], check=True)
sys.path.insert(0, '/content/LTSF-Linear')
import torch
from models import DLinear
from ptflops import get_model_complexity_info

dev = 'cuda' if torch.cuda.is_available() else 'cpu'
args = types.SimpleNamespace(seq_len=96, pred_len=720, individual=False, enc_in=321)
model = DLinear.Model(args).to(dev).eval()

n_params = sum(p.numel() for p in model.parameters())
macs, _ = get_model_complexity_info(model, (96, 321), as_strings=False,
                                    print_per_layer_stat=False, verbose=False)
print(f'device: {torch.cuda.get_device_name(0) if dev == "cuda" else "cpu"}')
print(f'params: {n_params:,} ({n_params/1e3:.1f}K)   MACs (batch 1): {macs/1e9:.3f}G')

def sync():
    if dev == 'cuda':
        torch.cuda.synchronize()

for bs in [1, 16, 32]:
    x = torch.randn(bs, 96, 321, device=dev)
    if dev == 'cuda':
        torch.cuda.empty_cache(); torch.cuda.reset_peak_memory_stats()
    with torch.no_grad():
        for _ in range(50):          # warm-up
            model(x)
        sync()
        runs = []
        for _ in range(5):           # 논문: 5회 평균
            t0 = time.perf_counter()
            for _ in range(200):
                model(x)
            sync()
            runs.append((time.perf_counter() - t0) / 200 * 1e3)
    line = f'batch {bs:>2}: {statistics.mean(runs):.3f} ms/step (5 runs: ' + ', '.join(f'{r:.3f}' for r in runs) + ')'
    if dev == 'cuda':
        line += f'   peak tensor memory: {torch.cuda.max_memory_allocated()/2**20:.1f} MiB'
    print(line)

if dev == 'cuda':
    used = subprocess.run(['nvidia-smi', '--query-gpu=memory.used', '--format=csv,noheader'],
                          capture_output=True, text=True).stdout.strip()
    print(f'nvidia-smi memory.used (CUDA context 포함): {used}')

## 실험 2. 미해결 칸을 2022년 5월 설정으로
대상: 다변량 Exchange 720, ILI 24·36·48·60, ETTh2 720. 설정마다 `--itr 5`.

In [ ]:
import sys, importlib, re
import numpy as np, pandas as pd
sys.path.insert(0, '/content'); import lab; importlib.reload(lab)
pd.set_option('display.width', 200)
LIN = 'scripts/EXP-LongForecasting/Linear/'
PAPER = {('Exchange', 720): 0.643, ('ILI', 24): 2.215, ('ILI', 36): 1.963, ('ILI', 48): 2.130,
         ('ILI', 60): 2.368, ('ETTh2', 720): 0.605}
OLD_LR = {'Exchange': '0.005', 'ILI': '0.05'}           # 2022-05 스크립트 값 (ETTh2는 지금과 같은 0.05)

def targets(setting):
    out = []
    for s in ['exchange_rate', 'ili', 'etth2']:
        for c in lab.commands(LIN + s + '.sh', [('--itr 1', '--itr 5')]):
            d = lab.info(c)
            if (d['dataset'], d['pred_len']) not in PAPER:
                continue
            if setting in ('B', 'C') and d['dataset'] in OLD_LR:
                lab.set_arg(c, '--learning_rate', OLD_LR[d['dataset']])
            if setting == 'B' and d['dataset'] not in OLD_LR:
                continue                                     # ETTh2는 학습률이 안 바뀌어서 B = A
            lab.set_arg(c, '--model_id', lab.arg(c, '--model_id') + '_' + setting)
            out.append(c)
    return out

def parse_dl(path):
    t = open(path, errors='ignore').read()
    return [float(x) for x in re.findall(r'mse_dl:([-\d.eE+]+)', t)]

RUNS = {}

In [ ]:
# A. 지금 스크립트 · 지금 코드
RUNS['A'] = targets('A')
lab.run(RUNS['A'], 'follow_A_now')

In [ ]:
# B. 2022-05 학습률 · 지금 코드
RUNS['B'] = targets('B')
lab.run(RUNS['B'], 'follow_B_oldlr')

In [ ]:
# C. 2022-05 학습률 · 2022-05 코드(1/L 초기화). 끝나면 원래 파일로 되돌린다.
!sed -i 's/^            # self.Linear_\(Seasonal\|Trend\).weight/            self.Linear_\1.weight/' models/DLinear.py
!grep -c "^            self.Linear_.*1/self.seq_len" models/DLinear.py
try:
    RUNS['C'] = targets('C')
    lab.run(RUNS['C'], 'follow_C_oldlr_init')
finally:
    !git checkout -q models/DLinear.py
!grep -c "^            # self.Linear_.*1/self.seq_len" models/DLinear.py

In [ ]:
# 결과 비교: 설정별 5회 MSE (지금 평가 / 논문 당시 평가)
rows = []
for s, cmds in RUNS.items():
    for c in cmds:
        d = lab.info(c)
        mses = [m for _, m, _ in lab.parse_log(c['logpath'])]
        dls = parse_dl(c['logpath'])
        for k, (m, dl) in enumerate(zip(mses, dls)):
            rows.append(dict(setting=s, dataset=d['dataset'], pred_len=d['pred_len'],
                             lr=lab.arg(c, '--learning_rate'), run=k, mse=m, mse_droplast=dl))
R = pd.DataFrame(rows)
g = R.groupby(['dataset', 'pred_len', 'setting'])
T = g.agg(lr=('lr', 'first'), n=('mse', 'size'),
          mean=('mse', 'mean'), std=('mse', 'std'), min=('mse', 'min'), max=('mse', 'max'),
          dl_mean=('mse_droplast', 'mean'), dl_min=('mse_droplast', 'min'), dl_max=('mse_droplast', 'max'))
T['paper'] = [PAPER[(d, p)] for d, p, _ in T.index]
T['in_range'] = (T['min'] <= T['paper']) & (T['paper'] <= T['max'])
T['dl_in_range'] = (T['dl_min'] <= T['paper']) & (T['paper'] <= T['dl_max'])
T.round(4)

## 3. 결과 복사용
아래 출력 두 덩어리(실험 1 출력은 위 셀)를 그대로 복사해서 붙여 주면 정리한다.

In [ ]:
print(R.to_csv(index=False, float_format='%.4f'))
os.makedirs('/content/extra_results', exist_ok=True)
R.to_csv('/content/extra_results/followup_unresolved.csv', index=False)